# Image Cleaning — Visual Inspection of Predictions per Store

Runs inference on **one image per store** (the most recent available) and displays the segmentation overlay so you can identify stores where the model prediction looks wrong.

**Workflow:**
1. Run all cells to generate the grid
2. Note the Store IDs where the prediction looks bad
3. Add those IDs to `STORES_TO_REMOVE` in the last cell
4. Run the last cell to save a cleaned version of the occupancy CSV

In [ ]:
import re
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image
import segmentation_models_pytorch as smp

NAIP_DIR       = Path("../../Data/naip_images")
CHECKPOINT_DIR = Path("../../checkpoints")
OCC_CSV        = Path("../../Data/naip_occupancy_pixel_ratio.csv")
CLEAN_CSV      = Path("../../Data/naip_occupancy_pixel_ratio_clean.csv")

FT_CKPT   = CHECKPOINT_DIR / "resnet34_unet_walmart_ft.pth"
BASE_CKPT = CHECKPOINT_DIR / "resnet34_unet_best.pth"
CKPT_PATH = FT_CKPT if FT_CKPT.exists() else BASE_CKPT

IMG_SIZE      = 256
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
CLASS_COLORS  = np.array([[0,0,0],[0,200,0],[220,50,50]], dtype=np.uint8)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CKPT_PATH.name}")

In [ ]:
model = smp.Unet(
    encoder_name="resnet34", encoder_weights=None,
    in_channels=3, classes=3, activation=None,
)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model = model.to(DEVICE).eval()
print("Model ready.")

In [ ]:
# One image per store — most recent date (last alphabetically given YYYY-MM-DD format)
all_paths = sorted(NAIP_DIR.glob("*.png"))

store_to_path = {}
for p in all_paths:
    m = re.match(r"^(\d+)_(\d{4}-\d{2}-\d{2})", p.stem)
    if m:
        store_id = int(m.group(1))
        store_to_path[store_id] = p   # later paths overwrite → keeps most recent

stores = sorted(store_to_path.keys())
print(f"Unique stores : {len(stores)}")
print(f"Example       : store {stores[0]} → {store_to_path[stores[0]].name}")

In [ ]:
preprocess = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

def infer_overlay(path: Path):
    orig    = np.array(Image.open(path).convert("RGB"))
    h, w    = orig.shape[:2]
    resized = np.array(Image.fromarray(orig).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    tensor  = preprocess(image=resized)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(tensor).argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    mask = np.array(Image.fromarray(pred).resize((w, h), Image.NEAREST))
    # blend overlay
    out = orig.copy().astype(float)
    for cls_id, color in enumerate(CLASS_COLORS):
        if cls_id == 0:
            continue
        where = mask == cls_id
        out[where] = out[where] * 0.5 + color * 0.5
    occ = int((mask == 2).sum()) / mask.size
    return out.astype(np.uint8), occ

results = {}
n = len(stores)
print(f"Running inference on {n} stores …")
for i, store_id in enumerate(stores, 1):
    overlay, occ = infer_overlay(store_to_path[store_id])
    results[store_id] = {"overlay": overlay, "occ": occ,
                         "date": store_to_path[store_id].stem.split("_")[1]}
    if i % 20 == 0 or i == n:
        print(f"  [{i:>2}/{n}]")
print("Done.")

In [ ]:
NCOLS = 6
NROWS = int(np.ceil(len(stores) / NCOLS))

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(NCOLS * 3.5, NROWS * 3.8))
axes = axes.flatten()

for i, store_id in enumerate(stores):
    rec = results[store_id]
    ax  = axes[i]
    ax.imshow(rec["overlay"])
    ax.set_title(
        f"Store {store_id}",
        fontsize=10, fontweight="bold", color="white",
        backgroundcolor="#2c3e50", pad=3
    )
    ax.set_xlabel(
        f"{rec['date']}  |  occ={rec['occ']*100:.2f}%",
        fontsize=7.5, labelpad=2
    )
    ax.set_xticks([])
    ax.set_yticks([])

for j in range(len(stores), len(axes)):
    axes[j].axis("off")

legend_patches = [
    mpatches.Patch(color=CLASS_COLORS[1]/255, label="empty"),
    mpatches.Patch(color=CLASS_COLORS[2]/255, label="occupied"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, 0.01))

fig.suptitle(
    "One Prediction per Store — Most Recent Image\n"
    "Note Store IDs where the prediction looks wrong",
    fontsize=13, fontweight="bold", y=1.002
)
plt.tight_layout()
plt.show()

## Mark Stores to Remove

Add the Store IDs you want to exclude to `STORES_TO_REMOVE` below, then run the cell.  
It will save a cleaned CSV without any images from those stores.

In [ ]:
# ── Fill in the store IDs to remove ────────────────────────────────────────
STORES_TO_REMOVE = [
    # e.g. 111, 874, 3047
]

# ── Apply filter and save ───────────────────────────────────────────────────
df = pd.read_csv(OCC_CSV, parse_dates=["date"])
df_clean = df[~df["store_id"].isin(STORES_TO_REMOVE)].reset_index(drop=True)

removed_images = len(df) - len(df_clean)
print(f"Stores removed : {len(STORES_TO_REMOVE)}  {STORES_TO_REMOVE}")
print(f"Images removed : {removed_images}")
print(f"Images kept    : {len(df_clean)} / {len(df)}")

df_clean.to_csv(CLEAN_CSV, index=False)
print(f"\nSaved → {CLEAN_CSV.resolve()}")